# Passeo-ai-Agent Hermes 7B Host\n\nNotebook ini meng-host `nous-hermes:7b` di Google Colab dan membuka endpoint HTTPS untuk dipakai oleh Passeo-ai-Agent.\n\nSetelah semua cell selesai, copy URL `https://...trycloudflare.com`, lalu paste ke `Set > Server model endpoint` di app.

In [ ]:
# 1. Install Ollama, Flask gateway, dan Cloudflare Tunnel\n!curl -fsSL https://ollama.com/install.sh | sh\n!pip -q install flask flask-cors requests\n!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared\n!chmod +x /usr/local/bin/cloudflared\n!ollama --version\n!cloudflared --version

In [ ]:
# 2. Start Ollama server dan download model 7B ke runtime Colab\nimport os, subprocess, time, requests\n\nMODEL = "nous-hermes:7b"\nenv = os.environ.copy()\nenv.update({\n    "OLLAMA_HOST": "127.0.0.1:11434",\n    "OLLAMA_CONTEXT_LENGTH": "512",\n    "OLLAMA_NUM_PARALLEL": "1",\n    "OLLAMA_MAX_LOADED_MODELS": "1",\n    "OLLAMA_KEEP_ALIVE": "10m"\n})\n\nollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)\nfor _ in range(60):\n    try:\n        if requests.get("http://127.0.0.1:11434/api/version", timeout=2).ok:\n            break\n    except Exception:\n        time.sleep(1)\nelse:\n    raise RuntimeError("Ollama server gagal start")\n\nprint("Ollama aktif. Download model 7B dimulai, ukuran sekitar 3.8GB.")\n!ollama pull nous-hermes:7b\n!ollama list

In [ ]:
# 3. Start Passeo gateway: /health, /api/chat, /v1/chat/completions\nimport json, threading\nfrom flask import Flask, Response, jsonify, request\nfrom flask_cors import CORS\n\napp = Flask("passeo-hermes-colab")\nCORS(app, resources={r"/*": {"origins": "*"}})\n\ndef normalize_payload(payload):\n    payload = dict(payload or {})\n    payload["model"] = payload.get("model") or MODEL\n    payload["stream"] = False\n    options = {"num_ctx": 512, "num_predict": 256, "num_thread": 2}\n    options.update(payload.get("options") or {})\n    payload["options"] = options\n    return payload\n\n@app.get("/")\n@app.get("/health")\n@app.get("/api/version")\ndef health():\n    try:\n        version = requests.get("http://127.0.0.1:11434/api/version", timeout=5).json()\n        return jsonify({"ok": True, "name": "Passeo Hermes 7B Colab", "model": MODEL, "ollama": version})\n    except Exception as exc:\n        return jsonify({"ok": False, "error": str(exc)}), 503\n\n@app.post("/api/chat")\ndef api_chat():\n    payload = normalize_payload(request.get_json(force=True, silent=True) or {})\n    upstream = requests.post("http://127.0.0.1:11434/api/chat", json=payload, timeout=900)\n    return Response(upstream.text, status=upstream.status_code, content_type=upstream.headers.get("content-type", "application/json"))\n\n@app.post("/v1/chat/completions")\ndef openai_chat():\n    payload = normalize_payload(request.get_json(force=True, silent=True) or {})\n    upstream = requests.post("http://127.0.0.1:11434/api/chat", json=payload, timeout=900)\n    if not upstream.ok:\n        return Response(upstream.text, status=upstream.status_code, content_type="application/json")\n    data = upstream.json()\n    text = (data.get("message") or {}).get("content") or data.get("response") or ""\n    return jsonify({\n        "id": "passeo-colab",\n        "object": "chat.completion",\n        "model": payload["model"],\n        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}]\n    })\n\nthreading.Thread(target=lambda: app.run(host="0.0.0.0", port=8080, debug=False, use_reloader=False), daemon=True).start()\ntime.sleep(2)\nprint(requests.get("http://127.0.0.1:8080/health", timeout=10).json())

In [ ]:
# 4. Buka public HTTPS tunnel. Copy URL trycloudflare.com ke app Passeo-ai-Agent.\nimport re, sys\n\ntunnel_proc = subprocess.Popen(\n    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8080", "--no-autoupdate"],\n    stdout=subprocess.PIPE,\n    stderr=subprocess.STDOUT,\n    text=True,\n    bufsize=1\n)\n\npublic_url = None\nfor line in tunnel_proc.stdout:\n    print(line, end="")\n    match = re.search(r"https://[-a-zA-Z0-9.]+trycloudflare.com", line)\n    if match:\n        public_url = match.group(0)\n        print("\\n=== PASSEO MODEL ENDPOINT ===")\n        print(public_url)\n        print("Paste URL ini ke Set > Server model endpoint, lalu klik Uji server 7B.")\n        break\n\nif not public_url:\n    raise RuntimeError("Tunnel URL tidak muncul. Run cell ini lagi.")

## Test manual\n\nSetelah URL tunnel muncul, test dari browser:\n\n```text\nhttps://URL-KAMU.trycloudflare.com/health\n```\n\nColab free bisa disconnect. Kalau runtime restart, run all lagi dan gunakan URL tunnel baru.